# Stefan Problem

This is a testcase for evaporation.  
Some liquid evaporated at a planar fluid vapor interface.  
The evaporation is driven by heat transfer through the vapor, originating from a heated wall.  
Initially $t = 0$ there is no vapor in the system.  
The simulation is started from an analytical solution at $t_0 > 0$.  
We can then compare the simulated interface position in time to the analytically predicted.  
See `10.1016/j.ijheatmasstransfer.2021.121233`  

Note the following:
* A deviation from the source is the considerably larger surface tension, to deal with capillary timestep restrictions.
* If we initialize with zero velocity fields the interface movement is off in the first timestep. There are two workarounds: initialize with the analytical velocity (done now) or use a very small first timestep.
* Paraview seems to not read time data correcty. Thus time annotation is missing in videos.

In [ ]:
#r ".\binaries\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [ ]:
string proj = "StefanProblem";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.DefaultDatabase.Sessions;
sessions

## Grid and Boundary/Initial Value configuration

Rescale temperature, for some reason better for the solver:  
$\Theta = \frac{T - T_{s}}{T_{s} - T_{Wall}}$  
$\Delta T = {T_{s} - T_{Wall}}$  

$ \tilde{c} = \Delta T * c$  
$ \tilde{k} = \Delta T * k$  

In [ ]:
static class Constants{
    // careful order of declaration matters!!!
    public static double L = 0.001;
    public static double Xi = 0.066916063714767; // see Matlab Sheet StefanProblem.m

    // material parameters
    public static double rho_A = 958.4;
    public static double rho_B = 0.597;

    public static double mu_A = 2.8e-4;
    public static double mu_B = 1.26e-5;

    public static double Sigma = 0.059 / 1000000.0; // very small surface tension so that capillary timestep is bigger...

    // Original 373.15 K + 10 K => 0 + 1
    public static double c_A = 4216.0 * 10;
    public static double c_B = 2030.0 * 10;

    public static double k_A = 0.679 * 10;
    public static double k_B = 0.025 * 10;

    public static double hVap  = 2.258e6;
    public static double T_sat = 0.0;//373.15;
    public static double T_wall = 1.0;//T_sat + 10.0;

    public static double alpha_B = k_B / (c_B*rho_B);
    public static double eps = 1.0-rho_B/rho_A;

    public static double x0 = 0.000105;
    public static double t0 = Math.Pow(0.5 * x0 / Xi, 2) / alpha_B; // start time, to avoid singular massflux at t=0
    public static double tE = 0.6 - t0; // Endtime
    // capillary timestep , for finest res + highest degree, use this, for comparability?!, Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) => 0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [ ]:
static class GridFactory{
    public static GridCommons Grid(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(0, Constants.L, res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, h, 2); // quasi 1D always use one cell
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "wall_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "pressure_outlet_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if(Math.Abs(X[0]) < 1e-6)
                return 2;
            if(Math.Abs(X[0] -  Constants.L) < 1e-6)
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }
}

In [ ]:
public static class BoundaryAndInitialValueFactory { 

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfacePos(){");
            stw.WriteLine("         return (X,t) => 2 * " + Constants.Xi + " * Math.Sqrt(" + Constants.alpha_B + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfaceVel(){");
            stw.WriteLine("         return (X,t) => " + Constants.Xi + " * " + Constants.alpha_B + " / Math.Sqrt(" + Constants.alpha_B + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){");
            stw.WriteLine("         return InterfacePos()(X,t) - X[0];");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureA(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + "; // always saturation temp.");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double VelocityA(double[] X, double t){");
            stw.WriteLine("         return " + Constants.eps + " * InterfaceVel()(X,t);");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureB(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + " + (" + Constants.T_sat + " - " + Constants.T_wall + ") / MathNet.Numerics.SpecialFunctions.Erf(" + Constants.Xi + ") * MathNet.Numerics.SpecialFunctions.Erf(X[0]/(2*Math.Sqrt(" + Constants.alpha_B + "*(t+" + Constants.t0 + "))));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureBBoundary(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureBInitial(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + " - (" + Constants.T_wall + " - " + Constants.T_sat + ") * X[0]/InterfacePos()(X,t); // linear interpolation works better to project initial value");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double VelocityB(double[] X, double t){");
            stw.WriteLine("         return 0.0;");
            stw.WriteLine("     }");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }
   
    static public Formula Get_VA() {
        return new Formula("BoundaryAndInitialValues.VelocityA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA() {
        return new Formula("BoundaryAndInitialValues.TemperatureA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB() {
        return new Formula("BoundaryAndInitialValues.TemperatureB", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB_Boundary() {
        return new Formula("BoundaryAndInitialValues.TemperatureBBoundary", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB_Initial() {
        return new Formula("BoundaryAndInitialValues.TemperatureBInitial", true, AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode:GetPrefixCode());
    }
}

## Begin Postprocessing

In [ ]:
//var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
sessions = sessions.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();
sessions.ForEach(s => display(s))

Lets sort the sessions by degree

In [ ]:
SortedDictionary<int, List<ISessionInfo>> groupedSessions = new SortedDictionary<int, List<ISessionInfo>>(sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList()));

In [ ]:
groupedSessions.Display();

Load Log Data

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));

### Interface Position

Evaluate the analyical solution for the interface position

In [ ]:
public static void subsample(Plot2Ddata plt, int n){
    foreach(var group in plt.dataGroups){
        var X = group.Abscissas;
        var Y = group.Values;
        int N = X.Count();
        List<double> newX = new List<double>();
        List<double> newY = new List<double>();
        
        newX.Add(X[0]);
        newY.Add(Y[0]);
        for(int i = 1; i < N - 1; i++){
            if((i % n) == 0){
                newX.Add(X[i]);
                newY.Add(Y[i]);
            }
        }
        newX.Add(X[N-1]);
        newY.Add(Y[N-1]);
        
        group.Abscissas = newX.ToArray();
        group.Values = newY.ToArray();
    }
}

In [ ]:
double[] times = GenericBlas.Linspace(0.0, Constants.tE + Constants.t0, 1000);
double[] yValues = new double[times.Length];
var PhiFuncF = BoundaryAndInitialValueFactory.Get_Phi();
Func<double, double> PhiFunc = t => {
    return PhiFuncF.Evaluate(new double[2], t);
};
for(int i = 0; i < times.Length; i++){
    yValues[i] = PhiFunc(times[i] - Constants.t0);
}
Plot2Ddata RefPos = new Plot2Ddata(times, yValues, "exact");
RefPos.dataGroups.First().Format = new PlotFormat("r-");

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    var plot = dat.ElementAt(0);
    plot.ModFormat();
    foreach(var grp in plot.dataGroups){
        grp.Name = grp.Name.Split('-').ElementAt(1).Replace("H", "n=");
        grp.Abscissas = grp.Abscissas.Select(x => x + Constants.t0).ToArray();
    }
    plot = plot.Merge(RefPos);
    plot.LegendAlignment = new string[] {"i", "t", "l"}; 
    plot.Ylabel = "Interface position [m]";
    plot.Xlabel = "Time [s]";
    plot.LabelFont = 16;
    plot.LabelTitleFont = 16;
    plot.Title = "p=" + p;
    plot.TitleFont = 24;

    var gp = plot.ToGnuplot();
    var p1 = gp.PlotSVG(); 
    
    plot.ShowLegend = false;
    plot.XrangeMin = Constants.tE - Constants.t0;
    plot.XrangeMax = Constants.tE;

    plot.YrangeMin = 0.99 * PhiFunc(plot.XrangeMin.Value);
    plot.YrangeMax = 1.01 * PhiFunc(plot.XrangeMax.Value);
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        //"   <div>" +
        //    p2.ToString() +
        //"   </div style='float:right;'>" +
        "</div>");
    // display(pp);
}

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));
data.ForEach(dat => dat.Value.ForEach(pp => subsample(pp, 50)));
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    var plot = dat.ElementAt(0);
    plot.ModFormat();
    foreach(var grp in plot.dataGroups){
        grp.Name = grp.Name.Split('-').ElementAt(1).Replace("H", "n=");
        grp.Abscissas = grp.Abscissas.Select(x => x + Constants.t0).ToArray();
    }
    plot = plot.Merge(RefPos);
    plot.LegendAlignment = new string[] {"i", "t", "l"}; 
    plot.Ylabel = "Interface position [m]";
    plot.Xlabel = "Time [s]";
    plot.LabelFont = 16;
    plot.LabelTitleFont = 16;
    plot.Title = "p=" + p;
    plot.TitleFont = 24;

    var gp = plot.ToGnuplot();
    var p1 = gp.PlotSVG(); 
    
    plot.ShowLegend = false;
    plot.XrangeMin = Constants.tE - Constants.t0;
    plot.XrangeMax = Constants.tE;

    plot.YrangeMin = 0.99 * PhiFunc(plot.XrangeMin.Value);
    plot.YrangeMax = 1.01 * PhiFunc(plot.XrangeMax.Value);
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        //"   <div>" +
        //    p2.ToString() +
        //"   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

Repeat the same for positional errors

In [ ]:
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    Plot2Ddata PositionErrorAbs = new Plot2Ddata();
    Plot2Ddata PositionErrorRel = new Plot2Ddata();
    foreach(var dataGroup in dat.ElementAt(0).dataGroups){
        double[] times = dataGroup.Abscissas;
        double[] yValuesAbs = new double[times.Length];
        double[] yValuesRel = new double[times.Length];
        for(int i = 0; i < times.Length; i++){
            double y = dataGroup.Values[i];
            double y_ex = PhiFunc(times[i]);
            yValuesAbs[i] = y - y_ex;
            yValuesRel[i] = (y - y_ex)/y_ex;

        }
        Plot2Ddata errDataAbs = new Plot2Ddata(times, yValuesAbs, "abs-err-" + dataGroup.Name);
        PositionErrorAbs      = PositionErrorAbs.Merge(errDataAbs);
        Plot2Ddata errDataRel = new Plot2Ddata(times, yValuesRel, "rel-err-" + dataGroup.Name);
        PositionErrorRel      = PositionErrorRel.Merge(errDataRel);

    }

    // absolute errors
    var plot = PositionErrorAbs;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p1 = plot.PlotNow(); 
    
    plot = PositionErrorRel;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

### Spatial Convergence

Evaluated at last timestep  
Plot Positional error over hmin

In [ ]:
double t_End = data.First().Value.ElementAt(0).dataGroups.First().Abscissas.Last();
double x_Exact = PhiFuncF.Evaluate(new double[2], t_End);

var p = new Plot2Ddata();
foreach(var (k, group) in groupedSessions){
    int i = 0;

    List<double> XValues = new List<double>();
    List<double> YValues = new List<double>();

    foreach(var s in group){
        display(s.Name);        
        var h = Convert.ToDouble(s.KeysAndQueries["Grid:hMin"]);
        var x = data[k].ElementAt(0).dataGroups[i].Values.Last();
        double err = Math.Abs(x - x_Exact);
        XValues.Add(h);
        YValues.Add(err);
        i++;
    }
    p = p.Merge(new Plot2Ddata(XValues, YValues, "P = " + k));    
}

p.ModFormat();
p.LogX = true;
p.LogY = true;

HtmlString pp = new HtmlString(
    "<div>" +  
    "   <div>" +
        p.PlotNow().ToString() +
    "   </div>" +
    "</div>");
display(pp);

Convergence as used by Bures&Sato:   
$E_1 = \frac{1}{t_{max}-t_0} \sum_i |\frac{x_i}{x_{ex}(t_i)} - 1|\Delta t_i$

In [ ]:
public static double ErrInGrowthConstant(ISessionInfo sess){
    var dat = (new[] { sess }).ToList().ReadLogDataForXNSE("StefanProblem").ElementAt(0);

    var PhiFuncF = BoundaryAndInitialValueFactory.Get_Phi();
    Func<double, double> PhiFunc = t => {
        return PhiFuncF.Evaluate(new double[2], t);
    };  

    var x_ex = dat.dataGroups.First().Abscissas.Select(t => PhiFunc(t)).ToArray();
    double dt = dat.dataGroups.First().Abscissas[1] - dat.dataGroups.First().Abscissas[0];

    double E1 = 0.0;
    for(int i = 0; i < x_ex.Length; i++){
        E1 += dt * Math.Abs(dat.dataGroups.First().Values[i]/x_ex[i] - 1.0);
    }
    E1 *= 1.0/Constants.tE;
    return E1;
}

public static IEnumerable<double> ErrInGrowthConstant( IEnumerable<ISessionInfo> sess){
    return sess.Select(s => ErrInGrowthConstant(s));
}

In [ ]:
var p = new Plot2Ddata();
foreach(var (k, group) in groupedSessions){
    int i = 0;

    List<double> XValues = new List<double>();
    List<double> YValues = new List<double>();

    foreach(var s in group){
        display(s.Name);        
        var h = Convert.ToDouble(s.KeysAndQueries["Grid:hMin"]);
        double err = ErrInGrowthConstant(s);
        XValues.Add(h);
        YValues.Add(err);
        i++;
    }
    p = p.Merge(new Plot2Ddata(XValues, YValues, "P = " + k));    
}

p.ModFormat();
p.LogX = true;
p.LogY = true;

HtmlString pp = new HtmlString(
    "<div>" +  
    "   <div>" +
        p.PlotNow().ToString() +
    "   </div>" +
    "</div>");
display(pp);

In [ ]:
double maxErr = p.dataGroups.Select(dg => dg.Values.Select(vv => Math.Abs(vv)).Max()).Max();
maxErr.Display();

Limit seems to be $\approx 5*10^{-4}$. We used the time step size $\Delta t \approx 1.4*10^{-4}$ and as will be visible below, the error is $\propto \Delta t$.  
Thus to further reduce the errors, we need to resolve finer in time.  We fixed our timestep, the reference (Bures & Sato) used a timestep based on some CFL condition (adaptive).  
Their minimum is $\Delta t \approx 1 * 10^{-4} $.  

In [ ]:
// dt = CFL * dx / u_i
double u_i = Constants.rho_B* 1.0/Math.Sqrt(0.03) * Math.Sqrt(Constants.alpha_B) * Constants.Xi * (1.0 / Constants.rho_B - 1.0/Constants.rho_A);
double CFL = 0.1;
double dx = 0.1 * 0.001 / 50.0;
CFL * dx / u_i

### Growthrate


In [ ]:
//var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
sessions = sessions.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();

Lets sort the sessions by degree

In [ ]:
SortedDictionary<int, List<ISessionInfo>> groupedSessions = new SortedDictionary<int, List<ISessionInfo>>(sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList()));

Load Log Data

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));

Calculate the $L1$ relative error in the computed growthconstant:  
$E_1 = \frac{1}{t_{max}-t_0}\sum_i|\frac{x_I}{x_{I,ex}}-1|\Delta t_i$  
or as we have equally spaced timesteps:  
$E_1 = \frac{1}{N}\sum_i|\frac{x_I}{x_{I,ex}}-1|$

In [ ]:
static Formula _AnalyticalInterfacePos = BoundaryAndInitialValueFactory.Get_Phi();
static Func<double, double> AnalyticalInterfacePos = t => {
    return _AnalyticalInterfacePos.Evaluate(new double[2], t);
};

public static double RelErrorGrowthConstant(Plot2Ddata.XYvalues InterfacePos){
    double err = 0.0;
    
    int N = InterfacePos.Abscissas.Length;
    for(int i = 0; i < N; i++){
        err += Math.Abs(InterfacePos.Values[i]/AnalyticalInterfacePos(InterfacePos.Abscissas[i]) - 1.0);
    }
    err = err/N;

    return err;
}

public static (string, double[], double[]) ReadReferenceCSV(string file){
    List<double> X = new List<double>();
    List<double> Y = new List<double>();
    string name;
    using(var reader = new StreamReader(file))
    {
        var line = reader.ReadLine();
        var values = line.Split(';');    
        name = "Bureš, Sato : " + values[1] + " problem";

        while (!reader.EndOfStream)
        {
            line = reader.ReadLine();
            values = line.Split(';');

            X.Add(Convert.ToDouble(values[0].Replace(',','.')));
            Y.Add(Convert.ToDouble(values[1].Replace(',','.')));
        }
    }
    return (name, X.ToArray(), Y.ToArray());
}

In [ ]:
data[1].ElementAt(0).PlotNow()

In [ ]:
var dd = data[1].ElementAt(0).dataGroups.First();
Math.Abs(AnalyticalInterfacePos(dd.Abscissas.Second()) - dd.Values.Second()).Display();
RelErrorGrowthConstant(dd).Display();

In [ ]:
data[1].ElementAt(0).dataGroups.First().Values.First().Display();
AnalyticalInterfacePos(data[1].ElementAt(0).dataGroups.First().Abscissas.First()).Display();

In [ ]:
Plot2Ddata GrowthConstantErrorRel = new Plot2Ddata();

foreach(int p in data.Keys.OrderBy(x => x)){
    if(p == 5) continue; // skip p=4
    var dat = data[p];

    double[] grid = new double[dat.ElementAt(0).dataGroups.Length];
    double[] error = new double[dat.ElementAt(0).dataGroups.Length];

    for(int i = 0; i < dat.ElementAt(0).dataGroups.Length; i++){
        var dataGroup = dat.ElementAt(0).dataGroups[i];
        grid[i] = 1.0/Convert.ToDouble(dataGroup.Name.Split('-')[1].Remove(0,1)) * 8; // normalize with base grid
        error[i] = RelErrorGrowthConstant(dataGroup) * 100.0; // in %

    }

    Plot2Ddata GrowthConstantErrorRel_p = new Plot2Ddata(grid, error, "k=" + p);
    GrowthConstantErrorRel      = GrowthConstantErrorRel.Merge(GrowthConstantErrorRel_p);

}
{
    var Reference = ReadReferenceCSV("BuresSato2021_StefanProblem_GrowthConstant.csv");
    Plot2Ddata GrowthConstantErrorRel_ref = new Plot2Ddata(Reference.Item2, Reference.Item3, Reference.Item1);
    GrowthConstantErrorRel      = GrowthConstantErrorRel.Merge(GrowthConstantErrorRel_ref);
}

var plot = GrowthConstantErrorRel;
plot.LegendAlignment = new string[] {"i", "t", "l"}; 
plot.Ylabel = "Relative error [%]";
plot.Xlabel = "1/Grid level [-]";
plot.LabelFont = 16;
plot.LabelTitleFont = 16;
plot.Title = "Relative error in growth constant";
plot.TitleFont = 24;

plot.LogX = true;
plot.XrangeMin = 0.02;
plot.XrangeMax = 2;
plot.LogX2 = true;
plot.X2rangeMin = 0.02;
plot.X2rangeMax = 2;

plot.LogY = true;
plot.YrangeMin = 0.01;
plot.YrangeMax = 1.0;

plot.ModFormat();
var gp = plot.ToGnuplot();
var p2 = gp.PlotSVG(); 

HtmlString pp = new HtmlString(
    "<div>" +
    "   <div>" +
        p2.ToString() +
    "   </div style='float:right;'>" +
    "</div>");
display(pp);

In [ ]:
KeyValuePair<int, double>[] refslopes = new KeyValuePair<int, double>[] {new KeyValuePair<int, double>(1, 0.3), new KeyValuePair<int, double>(2, 0), new KeyValuePair<int, double>(3, 0), new KeyValuePair<int, double>(4, 0)};
var slopes = plot.Regression().Take(4).ToDictionary(x=> Convert.ToInt32(x.Key.Split("=").ElementAt(1)),x=>x.Value);
slopes.Display();

Comparison of DOFs:  
BoSSS on finest grid (first timestep)

In [ ]:
groupedSessions[3]

In [ ]:
var s = groupedSessions[3].First();
(s.GetDOF("VelocityX")+s.GetDOF("VelocityY")+s.GetDOF("Pressure")+s.GetDOF("Temperature")).Display()

Bures,Sato 2021 (estimated)

In [ ]:
Math.Pow(50, 1) * Math.Pow(8, 1) * 4

### Temporal Convergence

In [ ]:
//var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "T");
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "T");
sessions = sessions.Where(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"]) == 3).ToList();
sessions = sessions.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();
sessions.ForEach(s => s.Display())

In [ ]:
Dictionary<int, List<ISessionInfo>> groupedSessions = sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList());

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));

In [ ]:
var p = new Plot2Ddata();
foreach(var (k, group) in groupedSessions){
    int i = 0;
    int n = 1;

    List<double> XValues = new List<double>();
    List<double> YValues = new List<double>();

    foreach(var s in group){
        s.Name.Display();   
        // double t_End = data[k].ElementAt(0).dataGroups[i].Abscissas.Last();
        double t_End = data[k].ElementAt(0).dataGroups[i].Abscissas.ElementAt(n*(int)Math.Pow(2,3-i) );
        double x_Exact = PhiFuncF.Evaluate(new double[2], t_End);   
        t_End.Display();  
        var h = Convert.ToDouble(s.KeysAndQueries["dtFixed"]);
        // var x = data[k].ElementAt(0).dataGroups[i].Values.Last();
        var x = data[k].ElementAt(0).dataGroups[i].Values.ElementAt(n*(int)Math.Pow(2,3-i) );
        double err = Math.Abs(x - x_Exact);
        XValues.Add(h);
        YValues.Add(err);
        i++;
    }
    p = p.Merge(new Plot2Ddata(XValues, YValues, "P = " + k));    
}

p.ModFormat();
p.LogX = true;
p.LogY = true;
p.AddDataGroup(" O(1) ", new[] {1e-5, 1e-3}, new[] {1e-8, 1e-6}, "r-");

HtmlString pp = new HtmlString(
    "<div>" +  
    "   <div>" +
        p.PlotNow().ToString() +
    "   </div>" +
    "</div>");
pp.Display();

Positional error over time

In [ ]:
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    Plot2Ddata PositionErrorAbs = new Plot2Ddata();
    Plot2Ddata PositionErrorRel = new Plot2Ddata();
    foreach(var dataGroup in dat.ElementAt(0).dataGroups){
        double[] times = dataGroup.Abscissas;
        double[] yValuesAbs = new double[times.Length];
        double[] yValuesRel = new double[times.Length];
        for(int i = 0; i < times.Length; i++){
            double y = dataGroup.Values[i];
            double y_ex = PhiFunc(times[i]);
            yValuesAbs[i] = y - y_ex;
            yValuesRel[i] = (y - y_ex)/y_ex;

        }
        Plot2Ddata errDataAbs = new Plot2Ddata(times, yValuesAbs, "abs-err-" + dataGroup.Name);
        PositionErrorAbs      = PositionErrorAbs.Merge(errDataAbs);
        Plot2Ddata errDataRel = new Plot2Ddata(times, yValuesRel, "rel-err-" + dataGroup.Name);
        PositionErrorRel      = PositionErrorRel.Merge(errDataRel);

    }

    // absolute errors
    var plot = PositionErrorAbs;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p1 = plot.PlotNow(); 
    
    plot = PositionErrorRel;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

## Assertion

In [ ]:
if(maxErr > 2E-3){
    throw new ApplicationException("Too large spatial error!");
}

In [ ]:
foreach(var kvp in refslopes){
    if(kvp.Value > slopes[kvp.Key]){
        throw new ApplicationException("Slope too low for polynomial order : " + kvp.Key);
    }
}